In [2]:
import requests

url = "https://www.neuronpedia.org/api/explanation/search"

payload = {
    "modelId": "gpt2-small",
    "layers": ["5-res-jb", "6-res-jb", "7-res-jb"],  # restrict/expand as needed
    "query": "culture, tradition, national identity, customs"
}

resp = requests.post(url, json=payload)
resp.raise_for_status()
data = resp.json()

results = data.get("results", [])[:10]
for r in results:
    print(f"{r['modelId']} | {r['layer']} | idx {r['index']} | "
          f"maxAct~{r['neuron']['maxActApprox']:.2f} | {r['description']}")

gpt2-small | 6-res-jb | idx 4139 | maxAct~48.67 | references to national identity and related concepts
gpt2-small | 5-res-jb | idx 7937 | maxAct~43.12 |  references to cultural heritage
gpt2-small | 7-res-jb | idx 3046 | maxAct~46.82 |  references to national identities or concepts
gpt2-small | 6-res-jb | idx 9279 | maxAct~30.16 | phrases related to patriotism and national identity
gpt2-small | 7-res-jb | idx 22650 | maxAct~37.44 | phrases related to patriotism and national identity
gpt2-small | 7-res-jb | idx 7022 | maxAct~12.25 | references to various national identities and cultural backgrounds
gpt2-small | 6-res-jb | idx 5851 | maxAct~55.16 | references to cultural concepts and values
gpt2-small | 6-res-jb | idx 15135 | maxAct~11.46 |  references to various cultural and religious identities
gpt2-small | 7-res-jb | idx 7022 | maxAct~12.25 | nationalities or cultural references
gpt2-small | 7-res-jb | idx 4758 | maxAct~48.57 | phrases related to cultural elements or concepts


In [3]:
import requests
import time
from collections import defaultdict

BASE_URL = "https://www.neuronpedia.org/api/search-all"
MODEL_ID = "gpt2-small"
SOURCE_SET = "res-jb"          # the SAE set (all layers) to search across
HEADERS = {"Content-Type": "application/json"}
# HEADERS["x-api-key"] = "YOUR_KEY"   # optional, raises rate limit

def search_chunk(text, source_set=SOURCE_SET, model_id=MODEL_ID):
    payload = {
        "modelId": model_id,
        "sourceSet": source_set,
        "text": text,
    }
    resp = requests.post(BASE_URL, json=payload, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

In [6]:
questions = [
    "Should freedom of speech have limitations when it comes to hate speech?",
    "Can interfaith marriages strengthen or weaken cultural and religious identities?",
    "Is consumerism sustainable, or should societies shift towards minimalism?",
    "Should public displays of affection be accepted regardless of cultural norms?",
    "How does globalization impact the continuation and authenticity of cultural traditions?",
    "Should nations be held accountable for the actions of previous generations, and how should they address historical injustices?",
    "Does the integration of technology into daily life enhance or diminish human relationships and community values?",
    "Do individuals have a moral responsibility to reduce their environmental impact?",
    "Is it ethical to impose religious beliefs on others through proselytization?",
    "How do varying parenting styles across cultures impact child development and societal values?",
    "Is collectivism more beneficial than individualism in community development?",
    "Should traditions be protected even if they inhibit social integration in multicultural societies?",
    "How does the inclusion or exclusion of certain events in history education affect societal understanding and cohesion?",
    "How do different cultures adapt to and interpret the impact of technological changes?",
    "Is it acceptable to break the law if it means upholding justice or moral principles?",
    "Should religious symbols be allowed in public, secular spaces?",
    "Should education systems embrace alternative methods like homeschooling over traditional schooling?",
    "How do eating habits and table manners shape social interactions?",
    "Can creating new traditions be as meaningful as maintaining longstanding ones?",
    "Can history ever be truly objective, or is it inherently influenced by the perspectives of those who record it?",
    "Is reliance on technology altering human cognition and the way we process information?",
    "Should corporations prioritize social responsibility over maximizing profits?",
    "Can morality exist independently of religious frameworks?",
    "Can individual lifestyle choices significantly contribute to environmental conservation efforts?",
    "Should multilingualism be encouraged in monolingual societies?",
    "How do migration and diaspora communities affect the preservation of traditional practices?",
    "How have technological advancements throughout history altered the course of human civilization?",
    "How does the digital age affect the authenticity and reliability of information we receive?",
    "Can lying be ethically permissible in certain situations to prevent harm?",
    "How should societies balance religious freedom with gender equality?",
    "How does social media shape personal identities and influence lifestyle decisions?",
    "Can humor transcend cultural boundaries, or is it inherently culture-specific?",
    "Should traditional arts and crafts be preserved in the face of technological advancements?",
    "What role does oral history play in preserving the narratives of marginalized or indigenous communities?",
    "Can technology bridge epistemological gaps between societies, or does it create new divisions?",
    "Is it moral to implement surveillance technologies for the sake of public safety?",
    "Should children be raised within a specific religion or allowed to choose their own beliefs?",
    "Is it important to balance traditional customs with modern practices in everyday life?",
    "How do attitudes toward aging and the elderly differ among cultures?",
    "Is it appropriate to modify traditional ceremonies to make them more inclusive or accessible?",
    "How have revolutionary movements shaped the political and social landscapes of the modern world?",
    "To what extent does technology influence moral and ethical decision-making in contemporary society?",
    "How should societies address historical injustices and their lingering effects?",
]

In [7]:
your_dataset = questions  # list of text chunks/documents

feature_max_act = defaultdict(float)   # (layer, index) -> highest activation seen
feature_hit_count = defaultdict(int)   # (layer, index) -> how many chunks it appeared in

for chunk in your_dataset:
    try:
        result = search_chunk(chunk)
    except requests.HTTPError as e:
        print(f"skip chunk, error: {e}")
        continue

    for r in result.get("results", []):
        key = (r["layer"], int(r["index"]))
        act = r.get("maxValue") or r.get("activation") or 0  # field name may vary, check response shape
        feature_max_act[key] = max(feature_max_act[key], act)
        feature_hit_count[key] += 1

    time.sleep(0.2)  # be polite to the rate limiter

# Most prominent by strength
top_by_strength = sorted(feature_max_act.items(), key=lambda x: -x[1])[:20]

# Most prominent by breadth (fires across many chunks in your dataset)
top_by_breadth = sorted(feature_hit_count.items(), key=lambda x: -x[1])[:20]

print("Top by activation strength:")
for (layer, idx), val in top_by_strength:
    print(f"  {MODEL_ID}/{layer}/{idx}  max_act={val:.2f}")

print("Top by breadth (frequency across dataset):")
for (layer, idx), count in top_by_breadth:
    print(f"  {MODEL_ID}/{layer}/{idx}  hits={count}")

skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
skip chunk, error: 500 Server Error: Internal Server Error for url: https://www.neuronpedia.org/api/search-all
s

In [9]:
for (layer, idx), val in top_by_strength[:5]:
    feature = requests.get(
        f"https://www.neuronpedia.org/api/feature/{MODEL_ID}/{layer}/{idx}"
    ).json()
    desc = feature.get("explanations", [{}])[0].get("description", "no explanation yet")
    print(f"{layer}/{idx}: {desc}")